# ASRA Phase 1 — Kaggle submission (ARC Prize 2026)

Matches the official sample pattern: write `my_agent.py` + `submission.parquet`, then Kaggle re-runs your agent on the competition API.

Reference: [ARC3 Sample Submission - Stochastic Goose](https://www.kaggle.com/code/inversion/arc3-sample-submission-stochastic-goose)

## 0. Bootstrap

In [ ]:
import glob
import os
import subprocess
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')
COMP_SLUG = 'arc-prize-2026-arc-agi-3'
WORKING = '/kaggle/working' if IS_KAGGLE else '.'

if IS_KAGGLE:
    COMP_CANDIDATES = [
        f'/kaggle/input/competitions/{COMP_SLUG}',
        f'/kaggle/input/{COMP_SLUG}',
    ]
    COMP_ROOT = next((p for p in COMP_CANDIDATES if os.path.isdir(p)), COMP_CANDIDATES[0])
else:
    COMP_ROOT = os.environ.get('ASRA_COMP_ROOT', 'private/kaggle-dataset/competition')

AGENTS_ROOT = os.path.join(COMP_ROOT, 'ARC-AGI-3-Agents')
WHEELS_DIR = os.path.join(COMP_ROOT, 'arc_agi_3_wheels')
ENV_DIR = os.path.join(COMP_ROOT, 'environment_files')
RECORDINGS_DIR = os.path.join(WORKING, 'recordings')
os.makedirs(RECORDINGS_DIR, exist_ok=True)

print('IS_KAGGLE:', IS_KAGGLE)
print('COMP_ROOT:', COMP_ROOT, '| exists:', os.path.isdir(COMP_ROOT))

if os.path.isdir(WHEELS_DIR):
    wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, '*.whl')))
    if wheels:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *wheels])
        print('Installed wheels:', len(wheels))
else:
    print('WARNING: wheels dir missing')

if os.path.isdir(AGENTS_ROOT) and AGENTS_ROOT not in sys.path:
    sys.path.insert(0, AGENTS_ROOT)

os.environ.setdefault('RECORDINGS_DIR', RECORDINGS_DIR)
os.environ.setdefault('ENVIRONMENTS_DIR', ENV_DIR if os.path.isdir(ENV_DIR) else 'environment_files')
os.environ['OPERATION_MODE'] = 'COMPETITION' if IS_KAGGLE else os.environ.get('OPERATION_MODE', 'OFFLINE')
os.environ['ASRA_AGENTS_ROOT'] = AGENTS_ROOT
print('OPERATION_MODE:', os.environ['OPERATION_MODE'])

## 1. Write `my_agent.py`

In [ ]:
MY_AGENT_CODE = '''"""ASRA Phase 1 agent for ARC Prize 2026 — ARC-AGI-3 (Kaggle Swarm)."""

from __future__ import annotations

import hashlib
import json
import os
import random
import sys
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent

SEED = int(os.environ.get("ASRA_SEED", "42"))
MAX_ACTIONS = int(os.environ.get("ASRA_MAX_ACTIONS", "80"))
SIMPLE_ACTIONS = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"]

random.seed(SEED)
np.random.seed(SEED)


def canonical_grid(grid: Any) -> List[List[int]]:
    arr = np.array(grid, dtype=int)
    return arr.tolist()


def state_hash(grid: Any) -> str:
    payload = json.dumps(canonical_grid(grid), separators=(",", ":"), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


class ActionSemanticsInferencer:
    def __init__(self) -> None:
        self.effects: Dict[Tuple[str, str], List[Dict[str, Any]]] = defaultdict(list)

    def observe(self, state_hash_value: str, action: str, diff: Dict[str, Any], reward: float) -> None:
        self.effects[(state_hash_value, action)].append(
            {"num_changed_cells": diff.get("num_changed_cells"), "reward": reward}
        )

    def infer(self, state_hash_value: str, action: str) -> Dict[str, Any]:
        effects = self.effects.get((state_hash_value, action), [])
        if not effects:
            return {"observations": 0, "hypothesis": "unknown", "consistency_score": None}
        counts = [e["num_changed_cells"] for e in effects if e["num_changed_cells"] is not None]
        std = float(np.std(counts)) if counts else 0.0
        mean = float(np.mean(counts)) if counts else None
        if mean == 0:
            hyp = "no-op / blocked"
        elif mean is not None and mean <= 1.5:
            hyp = "localized cell update"
        else:
            hyp = "multi-cell transform"
        return {
            "observations": len(effects),
            "hypothesis": hyp,
            "consistency_score": float(1.0 / (1.0 + std)) if counts else None,
        }


class ASRAExplorer:
    def __init__(self, action_names: List[str]) -> None:
        self.action_names = action_names
        self.state_action_counts: Counter = Counter()
        self.action_rewards: Dict[str, List[float]] = defaultdict(list)
        self.dead_ends: set = set()

    def update(self, state_hash_value: str, action: str, diff: Dict[str, Any], reward: float) -> None:
        self.state_action_counts[(state_hash_value, action)] += 1
        self.action_rewards[action].append(float(reward))
        if diff.get("num_changed_cells") == 0 and reward <= 0:
            self.dead_ends.add((state_hash_value, action))

    def choose_action(
        self,
        state_hash_value: str,
        semantics: ActionSemanticsInferencer,
        available: Optional[List[str]] = None,
    ) -> str:
        candidates = [a for a in self.action_names if available is None or a in available] or list(
            self.action_names
        )
        scores: Dict[str, float] = {}
        for action in candidates:
            if (state_hash_value, action) in self.dead_ends:
                continue
            sem = semantics.infer(state_hash_value, action)
            c = sem.get("consistency_score")
            uncertainty = 1.0 if c is None else (1.0 - min(1.0, c))
            local = self.state_action_counts[(state_hash_value, action)]
            mean_r = float(np.mean(self.action_rewards[action])) if self.action_rewards[action] else 0.0
            scores[action] = 2.0 / (1.0 + local) + 0.7 * uncertainty + 0.5 * mean_r + random.random() * 0.05
        return max(scores.items(), key=lambda kv: kv[1])[0] if scores else random.choice(candidates)


GLOBAL_SEMANTICS = ActionSemanticsInferencer()
GLOBAL_EXPLORER = ASRAExplorer(SIMPLE_ACTIONS)


class ASRAAgent(Agent):
    """Transition-centric explorer with state-conditioned action semantics."""

    MAX_ACTIONS = MAX_ACTIONS

    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN or self.action_counter >= self.MAX_ACTIONS

    def _available_simple(self, latest_frame: FrameData) -> List[str]:
        avail = getattr(latest_frame, "available_actions", None) or []
        names = [a.name for a in avail if hasattr(a, "name") and a.name in SIMPLE_ACTIONS]
        return names or SIMPLE_ACTIONS

    def _to_game_action(self, action_name: str, grid: Any) -> GameAction:
        ga = getattr(GameAction, action_name)
        if ga.is_complex():
            h, w = len(grid), len(grid[0]) if grid else 0
            ga.set_data({"x": w // 2, "y": h // 2})
        ga.reasoning = f"ASRA v0.3: {action_name}"
        return ga

    def choose_action(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        grid = latest_frame.frame
        name = GLOBAL_EXPLORER.choose_action(
            state_hash(grid), GLOBAL_SEMANTICS, self._available_simple(latest_frame)
        )
        return self._to_game_action(name, grid)

    def take_action(self, action: GameAction) -> Optional[FrameData]:
        self._last_action_name = action.name
        return super().take_action(action)

    def append_frame(self, frame: FrameData) -> None:
        super().append_frame(frame)
        if len(self.frames) < 2:
            return
        prev, curr = self.frames[-2], self.frames[-1]
        if not prev.frame or not curr.frame:
            return
        diff_count = int(np.sum(np.array(prev.frame) != np.array(curr.frame)))
        diff = {"num_changed_cells": diff_count}
        reward = float(getattr(curr, "levels_completed", 0) or 0)
        sh = state_hash(prev.frame)
        action = getattr(self, "_last_action_name", "UNKNOWN")
        GLOBAL_SEMANTICS.observe(sh, action, diff, reward)
        GLOBAL_EXPLORER.update(sh, action, diff, reward)


def _game_ids(environments: Any) -> List[str]:
    ids: List[str] = []
    for env in environments:
        if isinstance(env, str):
            ids.append(env)
        elif hasattr(env, "game_id"):
            ids.append(str(env.game_id))
        else:
            ids.append(str(env))
    return ids


def run_swarm() -> Any:
    """Play all competition games via official Swarm (called on Kaggle re-run)."""
    from agents import AVAILABLE_AGENTS
    from agents.swarm import Swarm
    from arc_agi import Arcade

    AVAILABLE_AGENTS["asra"] = ASRAAgent
    arcade = Arcade()
    games = _game_ids(arcade.get_environments())
    print(f"ASRA Swarm: {len(games)} games")
    swarm = Swarm(
        "asra",
        os.environ.get("ARC_BASE_URL", "https://three.arcprize.org"),
        games,
        tags=["asra-v0.3"],
    )
    return swarm.main()


if __name__ == "__main__":
    agents_root = os.environ.get("ASRA_AGENTS_ROOT", "")
    if agents_root and agents_root not in sys.path:
        sys.path.insert(0, agents_root)
    scorecard = run_swarm()
    if scorecard is not None:
        print("Scorecard score:", getattr(scorecard, "score", None))
'''

MY_AGENT_PATH = os.path.join(WORKING, 'my_agent.py')
with open(MY_AGENT_PATH, 'w', encoding='utf-8') as f:
    f.write(MY_AGENT_CODE)
print('Writing', MY_AGENT_PATH)

## 2. Write `submission.parquet` (Kaggle submit gate)

In [ ]:
import pandas as pd

SUBMISSION_PATH = os.path.join(WORKING, 'submission.parquet')
pd.DataFrame(
    [
        {
            'game_id': 'placeholder',
            'score': 0.0,
            'levels_completed': 0,
            'actions': 0,
            'completed': False,
            'agent': 'asra-v0.3',
        }
    ]
).to_parquet(SUBMISSION_PATH, index=False)

if IS_KAGGLE:
    print('Competition mode: wrote submission.parquet (validation gate)')
else:
    print('Non-competition mode: wrote dummy submission.parquet')
print('Output files:', os.listdir(WORKING))

## Done

Expected output: `my_agent.py` + `submission.parquet`.

Do **not** import or run Swarm in this notebook — Kaggle executes `my_agent.py` during scoring (see [discussion #694750](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3/discussion/694750)).